# Notebook 2 — Data Profiling & The LLM Defense Line

**Webinar 1: The AI-Ready Data Audit**

---

### Purpose

Profile the raw student dataset to understand its shape, sparsity, distributions,
and statistical anomalies — **before** any cleaning or imputation.

### Input / Output

| | File |
|---|---|
| Input | `data/student_data.xlsx` |
| Output | `data/profiled_data.xlsx` (raw data + profiling summary) |

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_DIR = os.path.join("..", "data")
input_file = os.path.join(DATA_DIR, "student_data.xlsx")

df = pd.read_excel(input_file, engine="openpyxl")

print("=" * 60)
print("  NOTEBOOK 2 — DATA PROFILING & THE LLM DEFENSE LINE")
print("=" * 60)
print(f"\n✅ Loaded: {input_file}")
print(f"   Shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")

---

## 🏛️ Architectural Deep-Dive: Why Tabular Cleaning Is the LLM's First Line of Defense

### The `teacher_notes` Problem

The `teacher_notes` column contains unstructured text — exactly the kind of data
that an enterprise **Retrieval-Augmented Generation (RAG)** pipeline would embed
into a Vector Database and feed to an LLM for question-answering.

But here is the trap: when the RAG retriever pulls a chunk, it also pulls the
**surrounding structured metadata** as context. If that metadata says
`attendance_pct = 141` or `marks_science = −10`, the LLM treats those as ground
truth and builds its answer around impossible facts.

### The Hallucination Chain

```
Dirty tabular data → embedded as metadata alongside teacher_notes
                   → LLM retrieves chunk with attendance = 141%
                   → generates: "This student has exceptional attendance
                      exceeding 100%, suggesting extra-curricular engagement"
                   → HALLUCINATION: the model invented a plausible
                      explanation for an impossible data-entry error
```

### The Defense

This entire tabular cleaning pipeline (Notebooks 2–4) is a **mandatory upstream
gate**. No row should reach the embedding layer until its structured columns have
been validated, imputed, and capped. Clean the table first; embed the text second.

In [ ]:
# ===========================================================================
# Structure & Type Overview
# ===========================================================================
print("\n── DataFrame .info() ──")
df.info()

print("\n── Data Types ──")
print(df.dtypes)

In [ ]:
# ===========================================================================
# Statistical Summary — exposes impossible values in min/max rows
# ===========================================================================
print("── Statistical Summary (.describe()) ──")
print(df.describe().to_string())

print(
    "\n📌 Look at the min/max rows above:"
    "\n   • marks_science min = -10  → impossible (valid range: 0–100)"
    "\n   • attendance_pct max = 141  → impossible (valid range: 0–100)"
    "\n   These will be flagged in Notebook 3 and capped in Notebook 4."
)

In [ ]:
# ===========================================================================
# Missing Values — matrix sparsity
# ===========================================================================
print("── Missing Values ──")
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(1)
missing_report = pd.DataFrame({"count": missing, "pct": missing_pct})
print(missing_report[missing_report["count"] > 0].to_string())
print(f"\n   Overall completeness: {(1 - df.isnull().mean().mean()) * 100:.1f}%")

In [ ]:
# ===========================================================================
# Unique Values — text columns (city, teacher_notes)
# ===========================================================================
print("── Unique Values (text columns) ──")
for col in df.select_dtypes(include="object").columns:
    print(f"\n   {col} → {df[col].nunique()} unique values")
    print(f"   {sorted(df[col].dropna().unique())}")

---

## 📊 The Kurtosis Trigger

### What Kurtosis Tells an Automated Pipeline

Kurtosis measures the **tailedness** of a distribution. A high Kurtosis
(leptokurtic, |k| > 3) means extreme outliers are present in the tails.

### Why This Matters for Imputation Strategy

In a production data pipeline (Airflow, Dagster, Prefect), you would compute
Kurtosis on every numeric column at ingest and compare it against a threshold.

When Kurtosis exceeds the threshold, the pipeline's branching logic should
**automatically switch** the imputation strategy:

| Kurtosis | Distribution Shape | Imputation Strategy |
|----------|-------------------|--------------------|
| ≤ 3 | Mesokurtic (normal-ish) | Mean is acceptable |
| > 3 | Leptokurtic (heavy tails) | **Force Median** — Mean is contaminated by outliers |

The `attendance_pct` column has the value 141 (an impossible outlier). Let's see
how that single value affects the Kurtosis and corrupts the Mean.

In [ ]:
# ===========================================================================
# Kurtosis Calculation — the pipeline circuit breaker
# ===========================================================================
att_kurtosis = df["attendance_pct"].kurtosis()
att_mean = df["attendance_pct"].mean()
att_median = df["attendance_pct"].median()

print("── Kurtosis Analysis: attendance_pct ──")
print(f"   Kurtosis : {att_kurtosis:.4f}")
print(f"   Mean     : {att_mean:.2f}")
print(f"   Median   : {att_median:.2f}")
print(f"   Delta    : {abs(att_mean - att_median):.2f} (Mean − Median gap)")

if abs(att_kurtosis) > 3:
    print(
        "\n⚠️  CIRCUIT BREAKER TRIGGERED: Kurtosis > 3"
        "\n   The distribution is leptokurtic — extreme outliers detected."
        "\n   Pipeline decision: ABANDON Mean → FORCE Median imputation."
    )
else:
    print(
        "\n✅ Kurtosis ≤ 3 — distribution tails are manageable."
        f"\n   However, the Mean/Median gap of {abs(att_mean - att_median):.2f}"
        "\n   still suggests skew. Median remains the safer imputation choice."
    )

print(
    "\n🔀 In production, this Kurtosis check would be an Airflow BranchPythonOperator"
    "\n   that routes to 'task_impute_median' or 'task_impute_mean' automatically."
)

In [ ]:
# ===========================================================================
# Reusable Profiling Summary Function
# ===========================================================================

def profile_dataset(data: pd.DataFrame) -> pd.DataFrame:
    """One-shot profiling summary reusable on any DataFrame."""
    return pd.DataFrame({
        "dtype": data.dtypes,
        "missing_count": data.isnull().sum(),
        "missing_pct": (data.isnull().mean() * 100).round(1),
        "unique_values": data.nunique(),
    })


profile = profile_dataset(df)
print("── Profiling Summary ──")
print(profile.to_string())

In [ ]:
# ===========================================================================
# Export — raw data + profile summary
# ===========================================================================
output_file = os.path.join(DATA_DIR, "profiled_data.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="raw_data", index=False)
    profile.to_excel(writer, sheet_name="profile_summary")

print(f"\n✅ Saved: {output_file}")
print(f"   Sheets: raw_data, profile_summary")
print(f"\n→ Next: Notebook 3 — QA Checks, Thresholds & Data Leakage")